# House Prices — 住宅価格予測（回帰入門）

Kaggle の定番回帰コンペ House Prices を題材に、対数変換から提出までの流れを学びます。

> **Kaggle Tutorial** 対応ノートブック（`/competitions/house-prices`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Kaggle Notebook 上では既存の /kaggle/input/house-prices-advanced-regression-techniques を使用し、
# Google Colab やローカル環境では互換スキーマの住宅価格サンプルデータを自動配置します。
import os
import numpy as np
import pandas as pd

def _ensure_house_prices_data():
    target_dir = "/kaggle/input/house-prices-advanced-regression-techniques"
    if os.path.exists(os.path.join(target_dir, "train.csv")):
        return

    try:
        os.makedirs(target_dir, exist_ok=True)
        data_dir = target_dir
    except OSError:
        data_dir = os.path.abspath("./.kaggle_input/house-prices-advanced-regression-techniques")
        os.makedirs(data_dir, exist_ok=True)
        _orig_read_csv = pd.read_csv

        def _compat_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/"):
                rel = filepath_or_buffer[len("/kaggle/input/"):]
                local_path = os.path.abspath(os.path.join("./.kaggle_input", rel))
                if os.path.exists(local_path):
                    filepath_or_buffer = local_path
            return _orig_read_csv(filepath_or_buffer, *args, **kwargs)

        pd.read_csv = _compat_read_csv

    rng = np.random.default_rng(42)

    def _make_split(n, start_id, with_target):
        overall_qual = rng.integers(3, 10, size=n)
        first_flr = rng.normal(1160, 320, size=n).clip(450, 3200).round().astype(int)
        second_flr = np.where(rng.random(n) > 0.45, rng.normal(650, 260, size=n).clip(0, 1800), 0).round().astype(int)
        bsmt_sf = rng.normal(1050, 350, size=n).clip(0, 3000).round()
        gr_liv_area = first_flr + second_flr
        garage_cars = rng.choice([0, 1, 2, 3], size=n, p=[0.06, 0.27, 0.55, 0.12])
        year_built = rng.integers(1950, 2010, size=n)
        year_remod = np.maximum(year_built, rng.integers(1960, 2011, size=n))
        yr_sold = rng.integers(2006, 2011, size=n)
        lot_frontage = rng.normal(70, 20, size=n).clip(21, 180).round(1)
        lot_frontage[rng.random(n) < 0.17] = np.nan
        bsmt_With_nan = bsmt_sf.copy()
        if not with_target:
            bsmt_With_nan[0] = np.nan

        neighborhood = rng.choice(["NAmes", "CollgCr", "OldTown", "Edwards", "Somerst", "NridgHt"], size=n)
        ms_zoning = rng.choice(["RL", "RM", "FV", "C (all)"], size=n, p=[0.78, 0.15, 0.05, 0.02]).astype(object)
        if not with_target:
            ms_zoning[0] = "RareZone"  # test のみに現れる未知カテゴリ（handle_unknown="ignore" 検証用）
        kitchen_qual = np.where(overall_qual >= 8, "Ex", np.where(overall_qual >= 6, "Gd", "TA")).astype(object)
        garage_type = np.where(garage_cars == 0, None, rng.choice(["Attchd", "Detchd", "BuiltIn"], size=n).astype(object))

        df = pd.DataFrame({
            "Id": np.arange(start_id, start_id + n),
            "MSSubClass": rng.choice([20, 60, 50, 120], size=n),
            "MSZoning": pd.Series(ms_zoning, dtype=object),
            "LotFrontage": lot_frontage,
            "LotArea": rng.lognormal(9.1, 0.4, size=n).round().astype(int),
            "Neighborhood": neighborhood,
            "OverallQual": overall_qual,
            "OverallCond": rng.integers(4, 9, size=n),
            "YearBuilt": year_built,
            "YearRemodAdd": year_remod,
            "TotalBsmtSF": bsmt_With_nan,
            "1stFlrSF": first_flr,
            "2ndFlrSF": second_flr,
            "GrLivArea": gr_liv_area,
            "FullBath": rng.choice([1, 2, 3], size=n, p=[0.45, 0.50, 0.05]),
            "KitchenQual": pd.Series(kitchen_qual, dtype=object),
            "GarageType": pd.Series(garage_type, dtype=object),
            "GarageCars": garage_cars,
            "YrSold": yr_sold,
        })
        if with_target:
            log_price = (
                10.8
                + 0.14 * overall_qual
                + 0.00028 * gr_liv_area
                + 0.00018 * np.nan_to_num(bsmt_With_nan, nan=0.0)
                + 0.08 * garage_cars
                + 0.003 * (year_built - 1970)
                + np.where(neighborhood == "NridgHt", 0.18, 0.0)
                + rng.normal(0.0, 0.11, size=n)
            )
            df["SalePrice"] = np.expm1(log_price).round().astype(int)
        return df

    _make_split(1460, 1, True).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    _make_split(1459, 1461, False).to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"House Prices dataset ready at: {data_dir}")

_ensure_house_prices_data()

## 概要

[House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)
は、住宅の特徴から **販売価格 `SalePrice` を予測する回帰**コンペです。
約 80 個の特徴量があり、欠損処理や特徴量エンジニアリングの練習に最適です。

> **Note:** 評価指標は **対数を取った価格の RMSE**（実質 RMSLE）。提出は `Id` と予測した `SalePrice` の 2 列 CSV です。

## データ

- 目的変数：`SalePrice`（販売価格、連続値・右に歪んだ分布）
- 代表的な特徴：`OverallQual`（品質）、`GrLivArea`（居住面積）、`GarageCars`、`TotalBsmtSF` など

## ベースラインの方針

目的変数を対数変換（`log1p`）し、欠損補完とカテゴリ数値化を `Pipeline` でまとめ、
回帰モデルで予測します。**予測後は `expm1` で価格スケールへ戻して提出**します。

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor

train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

y = np.log1p(train["SalePrice"])
X = train.drop(columns=["Id", "SalePrice"])
X_test = test.drop(columns=["Id"])

num = X.select_dtypes("number").columns
cat = X.select_dtypes("object").columns
pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore")),
    ]), cat),
])

model = Pipeline([("pre", pre), ("gbr", GradientBoostingRegressor(random_state=42))])
model.fit(X, y)
pred = np.expm1(model.predict(X_test))

submission = pd.DataFrame({"Id": test["Id"], "SalePrice": pred})
submission.to_csv("submission.csv", index=False)

In [ ]:
print("submission shape:", submission.shape)
submission.head()

## 提出

生成した `submission.csv` をコンペの **Submit Predictions** からアップロードすると、
Public リーダーボードにスコアが表示されます。

> **Tip:** 対数変換した目的変数を `expm1` で戻し忘れると、スコアが極端に悪化します。提出前に `submission.head()` で値の桁を必ず確認しましょう。

## 学習トラックで詳しく学ぶ

このコンペの取り組みは、学習トラックで段階的に解説しています。

- L3-05 House Prices：問題理解と EDA
- L3-06 欠損処理・特徴量・回帰モデル・提出
- L3-07 特徴量強化とアンサンブルの入口